Kaggle competition: https://www.kaggle.com/competitions/playground-series-s6e8/overview


In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
from utils.kaggle_mine import download_kaggle_dataset

# download kaggle dataset
download_kaggle_dataset(
    resource="playground-series-s6e8",
    resource_type="competition",
    unzip=True,
    delete_zip=True,
)


Checking destination: /Users/saifullah/Coding/ML/KaggleCompetitions/Predicting Smartphone Addiction/data
Authenticating with Kaggle...
Starting competition download: playground-series-s6e8
Saving to: /Users/saifullah/Coding/ML/KaggleCompetitions/Predicting Smartphone Addiction/data


100%|██████████| 19.7M/19.7M [00:03<00:00, 5.56MB/s]


Download finished.
Extracting: playground-series-s6e8.zip


Extracted: playground-series-s6e8.zip
Deleted ZIP: playground-series-s6e8.zip
Ready: /Users/saifullah/Coding/ML/KaggleCompetitions/Predicting Smartphone Addiction/data


PosixPath('/Users/saifullah/Coding/ML/KaggleCompetitions/Predicting Smartphone Addiction/data')

In [3]:
train_df = pd.read_csv("./data/train.csv")
test_df = pd.read_csv("./data/test.csv")
sample_submission = pd.read_csv('./data/sample_submission.csv')

In [4]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Submission columns:", sample_submission.columns.tolist())
print("\nTarget balance:")
print(train_df["addicted_label"].value_counts(normalize=True))


Train shape: (691369, 14)
Test shape: (296302, 13)
Submission columns: ['id', 'addicted_label']

Target balance:
addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64


In [5]:
TARGET = "addicted_label"
ID_COL = "id"

Start training

In [6]:
X = train_df.drop(columns=[TARGET, ID_COL], errors="ignore")
y = train_df[TARGET]


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42, stratify=y
)

In [8]:
# 4. Numeric columns: fill NaN with median
numeric_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

# 5. Categorical columns: fill NaN with "Unknown", then encode
categorical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

# 6. Apply correct preprocessing based on each column type
preprocessor = ColumnTransformer(transformers=[
    (
        "numeric",
        numeric_pipeline,
        selector(dtype_include="number")
    ),
    (
        "categorical",
        categorical_pipeline,
        selector(dtype_include=["object", "category", "bool", "string"])
    )
])

In [9]:
model =  RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )


In [10]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [ ]:
pipeline.fit(
    X_train,y_train
)